In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## Functions

This project required the development of six specialized scraping functions responsible for collecting match results, match events, league standings, historical standings progression, titles, and squad information. The architecture was designed to be modular and reusable, enabling data extraction from both completed and ongoing seasons through round-specific queries. Furthermore, the same functions can be easily adapted to other competitions available on Transfermarkt, as long as they share the same underlying page structure. Each function and its role within the project are presented in the following sections.

### 1) get_events()

The get_events() function extracts all match events from a specific round of a Transfermarkt competition. It identifies the team involved, the minute of the event, the player responsible, and classifies the event into predefined categories, including regular goals, penalty goals, own goals, missed penalties, and red cards. The function also generates unique identifiers for each event and accounts for different page layouts to ensure accurate data extraction. The resulting data is returned in a structured format, ready for analysis or conversion into a DataFrame.

In [3]:
def get_events(headers, league, n_season, n_round):
        # Variables Needed
        goals_list = []
        count_event = 0
        n_match = 0
        season_id = f'PL-{n_season}'
        
        # Inicializing Beautiful Soup
        url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
        response = requests.get(url, headers=headers)
        response.status_code
        soup = BeautifulSoup(response.content, "lxml")

        # Storing all match related data in a single list
        all_matches = soup.find_all('table', {'style':'border-top: 0 !important;'})
        
        
        for match in all_matches:
            # Creating match identifier
            n_match += 1
            match_id = f'M-{n_season}-{n_round:02d}-{n_match:02d}'
            
            # Storing all events data in a single list
            event = match.find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

            # List with the entire class necessary to get the home and away team's names
            gross_h_team = match.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
            gross_a_team = match.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

            # Checking for a possible forum buttom
            home_forum_check = gross_h_team.find('a').get('href')
            away_forum_check = gross_a_team.find('a').get('href')

            # Different ways to get the title depending if it has the forum buttom
            if 'forum' in home_forum_check and 'forum' in away_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            elif 'forum' in home_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find('a').get('title')
            elif 'forum' in away_forum_check:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            else:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find('a').get('title')

            # Access one by one all match related events
            for row in event:
                # Temporary list to store events of a single match
                temp = []
                temp.append(season_id)
                temp.append(match_id)

                # Creating event identifier
                count_event += 1
                event_id = f"E-{n_season}-{n_round:02d}-{count_event:04d}"
                temp.append(event_id)

                # Transfermarkt separates home and away team events
                # Home Team Events
                try: 
                    event_type = row.find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-links'}).string
                    temp.append(h_team)
                    temp.append(event_minute)
                
                # Away Team Events
                except: 
                    event_type = row.find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-rechts'}).string
                    temp.append(a_team)
                    temp.append(event_minute)

                # Event Type Information
                if event_type == 'icon-tor-formation': temp.append(1) # Normal Goal
                elif event_type == 'icon-elfmeter-formation': temp.append(2) # Penalty Goal
                elif event_type == 'icon-eigentor-formation': temp.append(3) # Own Goal
                elif event_type == 'icon-verschossener-elfmeter-formation': temp.append(-1) # Penalty Missed
                elif event_type == 'icon-rotekarte-formation': temp.append(-2) # Red Cards
                elif event_type == 'icon-gelbrotekarte-formation': temp.append(-3) # second yellow
                else: temp.append(0) # Exceptions

                # Player wich made the action
                player = row.find('a').get('title')
                temp.append(player)    

                # Inserting all events related to the match into the list
                goals_list.append(temp)

        goals_list.insert(0,['season_id', 'match_id', 'event_id','goal_score_team','goal_minute','goal_type', 'goal_scorer_name'])
        return goals_list

In [4]:
premier_events = []

for season in range(2025,2026):
    if season > 1994: season_round = 39
    else: season_round = 43

    for round in range(1,season_round+1):
        event_data = get_events(headers,'premier-league',season,round)
        premier_events.extend(event_data[1:])

df_events = pd.DataFrame(premier_events, columns=event_data[0])
display(df_events)

,season_id,match_id,event_id,goal_score_team,goal_minute,goal_type,goal_scorer_name
0,PL-2025,M-2025-01-01,E-2025-01-0001,Liverpool FC,37',1,Hugo Ekitiké
1,PL-2025,M-2025-01-01,E-2025-01-0002,Liverpool FC,49',1,Cody Gakpo
2,PL-2025,M-2025-01-01,E-2025-01-0003,AFC Bournemouth,64',1,Antoine Semenyo
3,PL-2025,M-2025-01-01,E-2025-01-0004,AFC Bournemouth,76',1,Antoine Semenyo
4,PL-2025,M-2025-01-01,E-2025-01-0005,Liverpool FC,88',1,Federico Chiesa
...,...,...,...,...,...,...,...
1099,PL-2025,M-2025-38-08,E-2025-38-0021,Chelsea FC,62',-3,Wesley Fofana
1100,PL-2025,M-2025-38-09,E-2025-38-0022,Tottenham Hotspur,43',1,João Palhinha
1101,PL-2025,M-2025-38-10,E-2025-38-0023,West Ham United,67',1,Taty Castellanos
1102,PL-2025,M-2025-38-10,E-2025-38-0024,West Ham United,79',1,Jarrod Bowen


### 2) get_match()

The get_match() function scrapes match-level information from a specific round of a Transfermarkt competition. It collects the participating teams, final score, match date, referee, attendance, and generates unique identifiers for the season, round, and match. The function also handles different page layouts caused by the presence of forum links, ensuring that team names are extracted correctly. Finally, the collected data is organized into a structured list, making it ready for further processing or conversion into a DataFrame.

In [5]:
def get_match(headers, league, n_season, n_round):
    all_rounds = []
    n_match = 0

    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")
    
    all_information = soup.find_all('table', {'style':'border-top: 0 !important;'})

    # Gathering Useful Information
    for row in all_information:
        temp = []
        n_match += 1

        season_key = f'PL-{n_season}'
        match_key = f'M-{n_season}-{n_round:02d}-{n_match:03d}'

        if n_round < 10: round_key = f'R-{n_season}-0' + str(n_round)
        else: round_key = f'R-{n_season}-' + str(n_round)

        temp.append(season_key)
        temp.append(round_key)
        temp.append(match_key)

        # List with the entire class necesaire to get the home and away team's names
        gross_home_team = row.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
        gross_away_team = row.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

        # Checking for a possible forum buttom
        home_forum_check = gross_home_team.find('a').get('href')
        away_forum_check = gross_away_team.find('a').get('href')

        # Different ways to get the title depending if it has the forum buttom
        if 'forum' in home_forum_check and 'forum' in away_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        elif 'forum' in home_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find('a').get('title')
        elif 'forum' in away_forum_check:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        else:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find('a').get('title')

        # Getting the final score
        final_score = row.find('span', {'class':'matchresult finished'}).string

        # Appending data from a single match together
        temp.append(home_team)
        temp.append(final_score)
        temp.append(away_team)

        # Storing adicional info separately, easier to extract right information
        adicional_info = row.find_all('td', {'class':'zentriert no-border'})

        for i, item in enumerate(adicional_info):
            if i == 2:
                text = item.get_text(" ", strip=True)
                try: 
                    attendance = text.split()[0]
                    temp.append(attendance)
                except: temp.append(text)
            else:
                day_ref = item.find('a').string
                temp.append(day_ref.strip())

        all_rounds.append(temp)

    all_rounds.insert(0,['season_id','round_id', 'match_id', 'home_team', 'final_score', 'away_team', 'date', 'referee', 'attendance'])
    return all_rounds

In [14]:
premier_matches = []

for season in range(2000,2002):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        matches_data = get_match(headers,'premier-league',season,round)
        premier_matches.extend(matches_data[1:])

df_premier_matches = pd.DataFrame(premier_matches, columns=matches_data[0])
display(df_premier_matches)

,season_id,round_id,match_id,home_team,final_score,away_team,date,referee,attendance
0,PL-2000,R-2000-01,M-2000-01-001,Charlton Athletic,4:0,Manchester City,19/08/2000,Rob Harris,20.043
1,PL-2000,R-2000-01,M-2000-01-002,Chelsea FC,4:2,West Ham United,19/08/2000,Graham Barber,34.914
2,PL-2000,R-2000-01,M-2000-01-003,Coventry City,1:3,Middlesbrough FC,19/08/2000,Barry Knight,20.624
3,PL-2000,R-2000-01,M-2000-01-004,Derby County,2:2,Southampton FC,19/08/2000,Andy D'Urso,27.223
4,PL-2000,R-2000-01,M-2000-01-005,Leeds United,2:0,Everton FC,19/08/2000,Dermot Gallagher,40.010
...,...,...,...,...,...,...,...,...,...
745,PL-2001,R-2001-38,M-2001-38-006,Leeds United,1:0,Middlesbrough FC,11/05/2002,Uriah Rennie,40.218
746,PL-2001,R-2001-38,M-2001-38-007,Leicester City,2:1,Tottenham Hotspur,11/05/2002,David Elleray,21.716
747,PL-2001,R-2001-38,M-2001-38-008,Sunderland AFC,1:1,Derby County,11/05/2002,Alan Wiley,47.989
748,PL-2001,R-2001-38,M-2001-38-009,Manchester United,0:0,Charlton Athletic,11/05/2002,Graham Poll,67.571


### 3) get_placements()

The get_placements() function retrieves the league standings for a specific round of a Transfermarkt competition. It extracts each team's position, matches played, wins, draws, losses, goals scored, goal difference, and total points. Additionally, the function generates unique identifiers for the season and round, organizing the collected information into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [12]:
def get_placements(headers, league, n_season, n_round):
    round_classification = []
    
    url = f'https://www.transfermarkt.com/{league}/spieltagtabelle/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
    response = requests.get(url,headers=headers)
    soup = BeautifulSoup(response.content,'lxml')

    info = soup.find_all('tbody')
    table_info = info[2].find_all('tr')

    for i,row in enumerate(table_info):
        temp = []

        season_key = f'PL-{n_season}'
        round_key = f'R-{n_season}-{n_round:02d}'
        
        temp.append(season_key)
        temp.append(round_key)

        placement = i+1
        team = row.find('a').get('title')

        temp.append(placement)
        temp.append(team)

        adicional_info = row.find_all('td', {'class':'zentriert'})

        for i, item in enumerate(adicional_info):
            if i == 0: continue
            temp.append(item.string)

        round_classification.append(temp)

    round_classification.insert(0,['season_id','round_id','placement','team_name','matches','wins','draws','losses','goals','goal_dif','points'])
    return round_classification

In [13]:
premier_placements = []

for season in range(2024,2025):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        placements_data = get_placements(headers,'premier-league', season, round)
        premier_placements.extend(placements_data[1:])

df_premier_placements = pd.DataFrame(premier_placements, columns=placements_data[0])
display(df_premier_placements)

,season_id,round_id,placement,team_name,matches,wins,draws,losses,goals,goal_dif,points
0,PL-2024,R-2024-01,1,Brighton & Hove Albion,1,1,0,0,3:0,3,3
1,PL-2024,R-2024-01,2,Arsenal FC,1,1,0,0,2:0,2,3
2,PL-2024,R-2024-01,3,Liverpool FC,1,1,0,0,2:0,2,3
3,PL-2024,R-2024-01,4,Manchester City,1,1,0,0,2:0,2,3
4,PL-2024,R-2024-01,5,Aston Villa,1,1,0,0,2:1,1,3
...,...,...,...,...,...,...,...,...,...,...,...
755,PL-2024,R-2024-38,16,Wolverhampton Wanderers,38,12,6,20,54:69,-15,42
756,PL-2024,R-2024-38,17,Tottenham Hotspur,38,11,5,22,64:65,-1,38
757,PL-2024,R-2024-38,18,Leicester City,38,6,7,25,33:80,-47,25
758,PL-2024,R-2024-38,19,Ipswich Town,38,4,10,24,36:82,-46,22


### 4) get_squad()

The get_squad() function extracts squad-related information for every team participating in a Transfermarkt competition during a given season. It retrieves each team's market value, squad size, average player age, and number of foreign players. The function also handles slight variations in the page structure when extracting market values, ensuring consistent results. All collected information is organized into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [15]:
def get_squad(headers, league, n_season):
    value = []

    url = f'https://www.transfermarkt.com/{league}/startseite/wettbewerb/GB1/plus/?saison_id={n_season}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")

    tables = soup.find_all('table', {'class':'items'})
    main_table = tables[0]

    even_info = main_table.find_all('tr', {'class':'even'})
    odd_info = main_table.find_all('tr', {'class':'odd'})

    info = odd_info + even_info

    season_key = f'PL-{n_season}'

    for row in info:
        temp = []

        temp.append(season_key)

        team_name = row.find('a').get('title')
        
        if row.find_all('a')[2].get('href') == '#': team_value = row.find_all('a')[-1].string
        else: team_value = row.find_all('a')[3].string

        temp.append(team_name)
        temp.append(team_value)

        squad_info = row.find_all('td', {'class':'zentriert'})
        for i, item in enumerate(squad_info):
            if i != 0: temp.append(item.string)

        value.append(temp)

    value.insert(0, ['season_id', 'team_name','team_value','team_squad','team_avg_age','team_foreigners'])
    return value

In [16]:
premier_squad = []

# transfermarkt only contains values for team_value from 2004
for season in range(2020,2026):
    squad_data = get_squad(headers,'premier-league',season)
    premier_squad.extend(squad_data[1:])

df_premier_squad = pd.DataFrame(premier_squad, columns=squad_data[0])
display(df_premier_squad)

,season_id,team_name,team_value,team_squad,team_avg_age,team_foreigners
0,PL-2020,Manchester City,€1.04bn,36,25.3,23
1,PL-2020,Chelsea FC,€889.20m,39,25.7,23
2,PL-2020,Tottenham Hotspur,€703.50m,41,25.2,24
3,PL-2020,Everton FC,€511.20m,44,25.2,27
4,PL-2020,Wolverhampton Wanderers,€450.30m,38,24.4,32
...,...,...,...,...,...,...
115,PL-2025,Aston Villa,€613.98m,44,25.1,22
116,PL-2025,West Ham United,€469.73m,40,26.6,26
117,PL-2025,Sunderland AFC,€446.68m,43,24.6,31
118,PL-2025,Fulham FC,€376.20m,30,27.4,23


### 5) get_title()

The get_title() function retrieves the historical champions of a Transfermarkt competition. For each title-winning season, it extracts the champion club and its manager, while also converting the season label into a standardized season identifier. In the case of the Premier League, the function considers only seasons from 1992 onward, when the competition adopted its current format. The collected data is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [17]:
def get_title(headers, league):
    titles = []

    url = f'https://www.transfermarkt.com/{league}/erfolge/wettbewerb/GB1'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")

    all_info = soup.find_all('tbody')
    info = all_info[0].find_all('tr')

    for row in info:
        temp = []

        season = row.find('td', {'class':'zentriert'}).string
        if season == '91/92': break # First season of the current format of the Premier League, maybe add a parameter to stop
        
        x = int(season.strip('/')[0] + season.strip('/')[1])
        if x > 90: n_season = x+1900
        else: n_season = x+2000
        season_key = f'PL-{n_season}'
        
        team_manager = row.find_all('a')
        temp.append(season_key)
        temp.append(season)

        for i, item in enumerate(team_manager):
            if i == 0: continue
            temp.append(item.string)
        
        titles.append(temp)

    titles.insert(0,['season_id', 'season_name','team_name', 'manager_name'])
    return titles

In [18]:
premier_titles = get_title(headers,'premier-league')

df_premier_titles = pd.DataFrame(premier_titles[1:], columns=premier_titles[0])
display(df_premier_titles)

,season_id,season_name,team_name,manager_name
0,PL-2025,25/26,Arsenal FC,Mikel Arteta
1,PL-2024,24/25,Liverpool FC,Arne Slot
2,PL-2023,23/24,Manchester City,Pep Guardiola
3,PL-2022,22/23,Manchester City,Pep Guardiola
4,PL-2021,21/22,Manchester City,Pep Guardiola
5,PL-2020,20/21,Manchester City,Pep Guardiola
6,PL-2019,19/20,Liverpool FC,Jürgen Klopp
7,PL-2018,18/19,Manchester City,Pep Guardiola
8,PL-2017,17/18,Manchester City,Pep Guardiola
9,PL-2016,16/17,Chelsea FC,Antonio Conte


### 6) get_table()

The get_table() function retrieves the final league table for a specific season of a Transfermarkt competition. It extracts each team's final position, matches played, wins, draws, losses, goals scored, goal difference, and total points. The function generates a standardized season identifier and organizes the collected information into a structured dataset, making it suitable for analysis or conversion into a DataFrame.

In [19]:
def get_table(headers, league, n_season):
    final_placement = []

    url = f'https://www.transfermarkt.com/{league}/tabelle/wettbewerb/GB1/saison_id/{n_season}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")

    all_info = soup.find_all('tbody')
    info = all_info[1].find_all('tr')

    season_key = f'PL-{n_season}'

    for i,row in enumerate(info):
        temp = []

        temp.append(season_key)

        position = i+1
        team = row.find('a').get('title')
        data_info = row.find_all('td', {'class':'zentriert'})

        temp.append(position)
        temp.append(team)

        for i, item in enumerate(data_info):
            if i == 0: continue
            temp.append(item.string)
        
        final_placement.append(temp)

    final_placement.insert(0,['season_id', 'pos','team_name','played','wins','draws','losses','goals','goal_dif','points'])
    return final_placement

In [20]:
premier_table = []

for season in range(2020,2026):
    table_data = get_table(headers,'premier-league',season)
    premier_table.extend(table_data[1:])

df_premier_table = pd.DataFrame(premier_table,columns=table_data[0])
display(df_premier_table)

,season_id,pos,team_name,played,wins,draws,losses,goals,goal_dif,points
0,PL-2020,1,Manchester City,38,27,5,6,83:32,51,86
1,PL-2020,2,Manchester United,38,21,11,6,73:44,29,74
2,PL-2020,3,Liverpool FC,38,20,9,9,68:42,26,69
3,PL-2020,4,Chelsea FC,38,19,10,9,58:36,22,67
4,PL-2020,5,Leicester City,38,20,6,12,68:50,18,66
...,...,...,...,...,...,...,...,...,...,...
115,PL-2025,16,Nottingham Forest,38,11,11,16,48:51,-3,44
116,PL-2025,17,Tottenham Hotspur,38,10,11,17,48:57,-9,41
117,PL-2025,18,West Ham United,38,10,9,19,46:65,-19,39
118,PL-2025,19,Burnley FC,38,4,10,24,38:75,-37,22
